# S06 Fine Labelling of All Lineages

Generic sub-clustering and relabelling of the broad annotation from step 05b. Each entry in
`lineages` defines the parent labels to pool, the Leiden resolution, marker panels and
the sub-cluster → label mapping. For every lineage the notebook re-clusters on the scVI
latent space, plots QC, marker and prediction evidence, and writes the new labels back.

To add or re-annotate a lineage: add an entry with `mapping=None`, run, read the evidence,
fill in `mapping` (and `expected_n_clusters`), re-run.

In [ ]:
# !pip install --quiet scanpy leidenalg igraph scikit-learn
# %cd <project>/notebooks

In [ ]:
import os

import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from sklearn.feature_extraction.text import TfidfTransformer

## Config

In [ ]:
input_path = "./../snRNAseq/output/cellbender/S5b_scVI_integrated_adata_cluster_latent10_logreg_major_cell_type.h5ad"

output_path = "./../snRNAseq/output/cellbender/S6_scVI_integrated_adata_cluster_latent10_logreg_fine_cell_type.h5ad"

figures_output_dir = "./../figures-cellbender/integration/3_annotation/2_fine"
tables_output_dir = "./../results/annotation/2_fine"
seed = 0


label_col = "cell_type"                 # overwritten with fine labels (downstream reads this column)
broad_col = "cell_type_broad"           # copy of the S05c labels; every run restarts from it
use_rep = "X_scVI_n_latent_10"
n_neighbors = 30
condition_col = "group2"                # shown in the QC panel (TN, NG, TN_d, TKIR)
qc_cols = ["phase", "scrublet_doublet", "n_genes", "percent_mito", condition_col]

n_marker_genes = 10
run_tfidf = True                        # TF-IDF marker dotplot per lineage (slow for the epithelial pool)
tfidf_cells_as_documents = False        # False reproduces the original (fitted on genes x cells)
marker_umaps = True                     # one UMAP per marker gene, in addition to the panel dotplot

os.makedirs(figures_output_dir, exist_ok=True)
os.makedirs(tables_output_dir, exist_ok=True)
sc.settings.figdir = figures_output_dir
sc.settings.verbosity = 1
sc.set_figure_params(dpi=80, figsize=(5, 5))

## 1. Load broad annotation

In [ ]:
adata = sc.read(input_path)
adata.uns.pop("log1p", None)
assert np.allclose(adata.X[:1000].data, np.round(adata.X[:1000].data)), "X is expected to hold raw counts"

if broad_col not in adata.obs:
    adata.obs[broad_col] = adata.obs[label_col].copy()
adata.obs[broad_col].value_counts()

## 2. Helpers

- `tfidf_markers`: TF-IDF on binarised counts + Wilcoxon ranking.
- `prediction_crosstab`: top logistic-regression (LuCA) predictions per group.
- `check_genes`: drops panel genes absent from the object and reports them.
- `refine`: sub-cluster one lineage, plot evidence, return new labels (or None).

In [ ]:
def tfidf_markers(adata, groupby, method="wilcoxon", cells_as_documents=tfidf_cells_as_documents):
    """Rank genes per group on TF-IDF-transformed binarised counts (X must hold counts)."""
    X_bin = (sparse.csr_matrix(adata.X) > 0).astype(np.float32)
    tfidf = (TfidfTransformer().fit_transform(X_bin) if cells_as_documents
             else TfidfTransformer().fit_transform(X_bin.T).T)
    bdata = sc.AnnData(X=sparse.csr_matrix(tfidf), obs=adata.obs[[groupby]].copy(), var=adata.var[[]].copy())
    bdata.obs[groupby] = bdata.obs[groupby].astype("category").cat.remove_unused_categories()
    sc.tl.rank_genes_groups(bdata, groupby=groupby, use_raw=False, method=method)
    return bdata


def prediction_crosstab(obs, groupby, top=3):
    """Top predicted labels (fraction of cells) and median probability per group."""
    ct = pd.crosstab(obs[groupby], obs["celltype_predictions"], normalize="index")
    top_labels = {g: "; ".join(f"{k} ({v:.0%})" for k, v in row.sort_values(ascending=False).head(top).items() if v > 0)
                  for g, row in ct.iterrows()}
    summary = pd.DataFrame({"n_cells": obs[groupby].value_counts(), "top_predictions": pd.Series(top_labels)})
    summary["median_probability"] = obs.groupby(groupby, observed=True)["probabilities"].median()
    return summary


def check_genes(panels, var_names, label=""):
    """Keep panel genes present in var_names; report the rest."""
    kept, missing = {}, []
    for name, genes in panels.items():
        present = [g for g in dict.fromkeys(genes) if g in var_names]
        missing += [g for g in genes if g not in var_names]
        if present:
            kept[name] = present
    if missing:
        print(f"{label}: not found, skipped -> {missing}")
    return kept


def refine(adata, name, parents, resolution, markers, mapping=None, default=None,
           expected_n_clusters=None, apply=True):
    """Sub-cluster `parents`, save evidence, and return new labels (None if nothing to apply)."""
    sub = adata[adata.obs[broad_col].isin(parents)].copy()
    if sub.n_obs <= n_neighbors:
        raise ValueError(f"{name}: {sub.n_obs} cells, too few to sub-cluster")
    sc.pp.normalize_total(sub, target_sum=1e4)
    sc.pp.log1p(sub)
    sc.pp.neighbors(sub, n_neighbors=n_neighbors, use_rep=use_rep, random_state=seed)
    sc.tl.umap(sub, random_state=seed)
    sc.tl.leiden(sub, key_added="leiden_sub", resolution=resolution, flavor="leidenalg", random_state=seed)
    n_clusters = sub.obs["leiden_sub"].nunique()
    print(f"{name}: {sub.n_obs} cells, {n_clusters} sub-clusters")

    # QC + composition
    sc.pl.umap(sub, color=["leiden_sub", broad_col, "celltype_predictions"] + qc_cols, ncols=4, wspace=0.4,
               size=15, color_map="OrRd", legend_fontsize="xx-small", save=f"_{name}_subclusters_qc.pdf")

    # markers: data-driven (logreg, TF-IDF) and curated panels
    sc.tl.rank_genes_groups(sub, "leiden_sub", method="logreg", key_added="logreg", max_iter=1000)
    sc.pl.rank_genes_groups_dotplot(sub, n_genes=n_marker_genes, key="logreg", groupby="leiden_sub",
                                    standard_scale="var", dendrogram=False, save=f"{name}_logreg.pdf")
    if run_tfidf:
        counts = sc.AnnData(X=adata[sub.obs_names].X, obs=sub.obs[["leiden_sub"]], var=adata.var[[]])
        sc.pl.rank_genes_groups_dotplot(tfidf_markers(counts, "leiden_sub"), n_genes=n_marker_genes,
                                        dendrogram=False, standard_scale="var", save=f"{name}_tfidf.pdf")
    panels = check_genes(markers, sub.var_names, name)
    sc.pl.dotplot(sub, panels, groupby="leiden_sub", standard_scale="var", save=f"{name}_marker_panels.pdf")
    if marker_umaps:
        genes = list(dict.fromkeys(g for v in panels.values() for g in v))
        sc.pl.umap(sub, color=["leiden_sub"] + genes, use_raw=False, color_map="OrRd", size=25,
                   legend_loc="on data", legend_fontsize="small", save=f"_{name}_markers.pdf")

    summary = prediction_crosstab(sub.obs, "leiden_sub")
    new_labels = None
    if mapping is not None:
        clusters = sub.obs["leiden_sub"].astype(str)
        unmapped = sorted(set(clusters) - set(mapping), key=int)
        if expected_n_clusters is not None and n_clusters != expected_n_clusters:
            msg = f"{name}: {n_clusters} sub-clusters, mapping written for {expected_n_clusters}"
        elif unmapped and default is None:
            msg = f"{name}: sub-clusters without a label: {unmapped}"
        else:
            msg = None
        summary["new_label"] = summary.index.map(lambda c: mapping.get(c, default))
        if msg and apply:
            raise ValueError(msg)
        if msg:
            print("draft mapping incomplete:", msg)
        if apply:
            new_labels = clusters.map(lambda c: mapping.get(c, default))
            sub.obs["new_label"] = new_labels.astype("category")
            sc.pl.umap(sub, color="new_label", size=15, legend_fontsize="x-small", legend_fontoutline=True,
                       title=f"{name}: fine labels", save=f"_{name}_annotated.pdf")
    summary.to_csv(f"{tables_output_dir}/{name}_subcluster_summary.csv")
    display(summary)
    return new_labels

## 3. Refine each lineage

Restarts from `broad_col`, so the cell can be re-run without compounding labels.
Lineages with `apply=False` are plotted for review only.

In [ ]:
# Note: for each lineage- do iterative sub-clustering and mapping:
#   parents              broad labels pooled and re-clustered
#   resolution           Leiden resolution
#   markers              {panel name: genes}; genes absent from var are reported and skipped
#   mapping              {sub-cluster: label}; None = explore only (no labels written)
#   default              label for sub-clusters not in mapping (None = every cluster must be mapped)
#   expected_n_clusters  guard against Leiden IDs drifting; None to skip
#   apply                False keeps a draft mapping without writing it
lineages = {
    "epithelial": dict(
        parents=["Epithelial+Tumor cells", "Epithelial+Tumor cells+Doublets"],
        resolution=1.0,                  # 0.6 lost ~13,000 nuclei to ambiguous clusters
        markers={
            "LUSC": ["KRT5", "KRT6A", "TP63", "NTRK2", "SOX2", "KRT17", "AKR1C1", "GPX2"],
            "LUAD": ["MUC1", "NAPSA", "NKX2-1", "SFTA2", "SFTA3", "KRT7"],
            "AT1": ["AGER", "CLDN18"],
            "AT2": ["SFTPC", "SFTPB"],
            "Stem": ["CD24", "CD44", "CSF3R", "CD36"],
            "Neuroendocrine": ["CHGA", "SYP", "NCAM1", "TUBA1A", "NEUROD1", "YAP1", "ASCL1", "NRCAM"],
            "EMT": ["VIM", "CDH1", "SERPINE1", "ZEB1", "CALD1"],
            "Proliferating": ["MKI67", "CDK1", "TOP2A", "STMN1", "PCLAF", "TK1"],
            "Undifferentiated": ["AGR2", "TACSTD2"],
            "Ciliated": ["PIFO", "FOXJ1", "HYDIN"],
            "Hepatocyte": ["AFP", "TTR", "CYP3A4", "ASGR1", "ASGR2", "FGB"],
            "Cholangiocyte": ["KRT19", "SOX9", "CFTR"],
            "Resistance-associated": ["NRXN3", "ERBB4", "RORA", "ELF3", "MET", "EGFR", "LRRK2", "DEPTOR"],
            "Other": ["ALPP", "CEACAM5", "CEACAM1", "ALDH2", "SIRT1", "ALDH1A1", "PIK3C2G", "SEMA3E",
                      "SPINK1", "HLA-DRB5", "NUPR1", "NTS", "IDO1", "ASPH", "EGF", "CD74", "CD274"],
        },
        mapping={"6": "Epithelial dividing", "19": "lowQC", "20": "Non-adeno", "23": "Non-adeno"},
        default="Adeno", expected_n_clusters=31, apply=True,
    ),
    "myeloid": dict(
        parents=["Myeloid"],
        resolution=1.0,
        markers={
            "cDC2": ["SIRPA", "ITGAX", "ITGAM", "CD1A", "CD1C", "CD207", "CLEC4A"],
            "cDC1": ["CLEC9A", "THBD", "XCR1", "CADM1"],
            "Migratory DC": ["CCR7", "BIRC3"],
            "Macrophage": ["APOE", "CD14", "MARCO"],
            "Alveolar macrophage": ["FABP4", "INHBA", "PPARG", "CD36", "STAB1"],
        },
        mapping={"17": "lowQC"}, default="Myeloid", expected_n_clusters=18, apply=True,
    ),
    "lymphoid": dict(
        parents=["Lymphoid", "B cell", "Plasma cell"],
        resolution=1.0,
        markers={
            "Naive T": ["IL7R", "TCF7", "SELL", "LEF1"],
            "Proliferating": ["CENPP", "MKI67", "TOP2A", "AFF3"],
            "Treg": ["FOXP3", "IL2RA"],
            "Inhibitory receptors": ["LAG3", "TIGIT", "PDCD1", "CTLA4", "HAVCR2"],
            "NK / cytotoxic": ["ITGAE", "NCAM1", "NKG7", "KLRD1", "GNLY"],
            "T lineage": ["CD8A", "CD8B", "CD4"],
        },
        # draft, not applied in the original (covers sub-clusters 0-10; 14 were observed);
        # lymphoid states are annotated in the dedicated lymphoid notebooks
        mapping={"0": "T cell regulatory", "1": "T cell CD4", "2": "T cell CD4", "3": "T cell CD4",
                 "4": "T cell CD4", "5": "T cell CD8", "6": "T cell NK-like", "7": "T cell CD8",
                 "8": "NK cell", "9": "T cell CD8", "10": "T cell CD8"},
        default=None, expected_n_clusters=None, apply=False,
    ),
    "endothelial": dict(
        parents=["Endothelial"],
        resolution=0.5,
        markers={
            "Lymphatic": ["LYVE1", "PROX1", "FLT4", "PDPN"],
            "Venous": ["EPHB4", "IL1R1"],
            "Post-capillary venule": ["ACKR1", "SELP"],
            "Arterial": ["GJA5", "IL33", "SOX17"],
            "Tip cell": ["PGF", "CXCR4"],
            "Pan-endothelial": ["CD34", "KIT", "KDR"],
        },
        # draft, not applied in the original
        mapping={"0": "Endothelial cell capillary", "1": "Endothelial cell venous",
                 "2": "Endothelial cell arterial", "3": "Endothelial cell lymphatic",
                 "4": "Endothelial cell capillary", "5": "Endothelial cell venous", "6": "lowQC",
                 "7": "Endothelial cell capillary"},
        default=None, expected_n_clusters=8, apply=False,
    ),
    "alveolar_club": dict(
        parents=["Alveolar cell type I", "Alveolar cell type II", "Club_Goblet cells"],
        resolution=0.6,
        markers={
            "Alveolar": ["AGER", "SFTPD", "SFTPC", "NAPSA", "NKX2-1"],
            "Tumour-associated": ["EGFR", "MET", "TP63", "KRT8", "KRT17", "LAMC2", "CEACAM5", "MUC21"],
            "Club/Goblet": ["SCGB1A1", "KRT7", "SCGB3A1", "SCGB3A2", "BPIFB1", "FOXA2"],
            "Ionocyte": ["CFTR", "FOXI1"],
            "Tuft": ["TRPM5", "POU2F3", "DCLK1"],
            "Neuroendocrine": ["PSMD5", "NGF", "ASCL1", "CHGA", "NEUROD1", "YAP1"],
            "Basal": ["KRT5"],
        },
        mapping={"0": "Alveolar cell type II", "1": "Alveolar cell type I", "2": "Alveolar cell type II",
                 "3": "Alveolar cell type II", "4": "Alveolar cell type I", "5": "Alveolar cell type I",
                 "6": "Transitional", "7": "Alveolar cell type II", "8": "Club_Goblet cells",
                 "9": "Transitional", "10": "Transitional"},
        default=None, expected_n_clusters=11, apply=True,
    ),
    "ciliated": dict(
        parents=["Ciliated epithelial"],
        resolution=0.5,
        markers={
            "Ciliated": ["FOXJ1", "HYDIN", "PIFO"],
            "Airway / alveolar": ["SCGB3A1", "SCGB3A2", "NAPSA", "NKX2-1", "SFTPC", "SFTPB"],
            "Tumour": ["CEACAM5", "CEACAM1"],
            "Fusion partners": ["ROS1", "EML4", "ALK"],
            "Immune": ["PTPRC"],
        },
        mapping={"0": "Ciliated", "1": "Ciliated", "2": "Ciliated", "3": "Ciliated", "4": "lowQC",
                 "5": "Ciliated",            # ROS1-high
                 "6": "Ciliated", "7": "Doublets", "8": "Ciliated", "9": "Ciliated"},
        default=None, expected_n_clusters=10, apply=True,
    ),
}

In [ ]:
fine = adata.obs[broad_col].astype(str).copy()
for name, spec in lineages.items():
    print(f"=== {name} ===")
    new_labels = refine(adata, name, **spec)
    if new_labels is not None:
        fine.loc[new_labels.index] = new_labels
adata.obs[label_col] = fine.astype("category")

transitions = pd.crosstab(adata.obs[broad_col], adata.obs[label_col])
transitions.to_csv(f"{tables_output_dir}/broad_to_fine_label_counts.csv")
transitions

## 4. Final labels and save (X = raw counts)

In [ ]:
sc.pl.umap(adata, color=label_col, size=15, legend_fontsize="x-small", legend_fontoutline=True,
           save="_annotated_major_epi_cell_type_fine.pdf")

summary = prediction_crosstab(adata.obs, label_col)
summary.to_csv(f"{tables_output_dir}/fine_label_prediction_summary.csv")

if save_unclean:
    adata.write(output_path)
summary

## 5. Remove low-quality nuclei and doublets

Removes nuclei with a fine label in `remove_labels` and nuclei called doublets by Scrublet.
The per-sample table shows whether removal is concentrated in particular samples.

In [ ]:
is_doublet = (adata.obs[doublet_col].astype(str) == "True") if doublet_col else pd.Series(False, index=adata.obs_names)
reason = pd.Series("kept", index=adata.obs_names)
reason[is_doublet] = "scrublet_doublet"
reason[adata.obs[label_col].isin(remove_labels)] = adata.obs[label_col].astype(str)   # label takes precedence
adata.obs["cleanup"] = pd.Categorical(reason)

sc.pl.umap(adata, color=[label_col, "cleanup"], size=15, legend_fontsize="x-small", wspace=0.5,
           save="_unclean_cell_type_cleanup.pdf")

removal = pd.crosstab(adata.obs["sample"], adata.obs["cleanup"], margins=True)
removal["pct_removed"] = (100 * (1 - removal["kept"] / removal["All"])).round(1)
removal.to_csv(f"{tables_output_dir}/cleanup_removed_per_sample.csv")
display(pd.crosstab(adata.obs[label_col], adata.obs["cleanup"]))
removal.sort_values("pct_removed", ascending=False)

In [ ]:
adata = adata[adata.obs["cleanup"] == "kept"].copy()
adata.obs[label_col] = adata.obs[label_col].cat.remove_unused_categories()
print(f"kept {adata.n_obs} nuclei")

## 6. Re-embed the clean object

New kNN graph, UMAP and Leiden on the scVI latent space (the model itself is not
retrained here; that is the second scVI run).

In [ ]:
sc.pp.neighbors(adata, n_neighbors=clean_n_neighbors, use_rep=use_rep, random_state=seed)
sc.tl.umap(adata, random_state=seed)
sc.tl.leiden(adata, key_added=clean_cluster_key, resolution=clean_leiden_resolution,
             flavor="leidenalg", random_state=seed)

sc.pl.umap(adata, color=label_col, size=10, legend_loc="on data", legend_fontsize="xx-small",
           save="_clean_cell_type_fine.pdf")
sc.pl.umap(adata, color=[clean_cluster_key, "scrublet_cluster_score", "scrublet_score"], size=35,
           color_map="OrRd", save="_clean_leiden_scrublet.pdf")
pd.crosstab(adata.obs[clean_cluster_key], adata.obs[label_col])

## 7. Save clean object (X = raw counts)

In [ ]:
if "raw_counts" in adata.layers:
    adata.X = adata.layers["raw_counts"].copy()
adata.write(clean_output_path)
adata